In [ ]:
import sys, os
print(os.getcwd())
sys.path.append("../src")
from load_nerf_data import load_nerf_data

In [ ]:
# Load and inspect LEGO dataset
imgs, sils, cams = load_nerf_data("../data/lego", split="train", resize_to=(200, 200))
# Print basic information about the dataset
print(imgs.shape, sils.shape, imgs.min().item(), imgs.max().item())
# Print camera center and background colour
print(cams.get_camera_center()[:3])
print("background colour:", imgs[0][sils[0] < 0.1].mean(0))

In [ ]:
import json
# Load and inspect LEGO dataset's camera transforms
with open("../data/lego/transforms_train.json") as f:
    t = json.load(f)
for fr in t["frames"][:3]:
    print([row[3] for row in fr["transform_matrix"][:3]])

In [ ]:
import torch
# Create a batch of points at the origin and transform them to screen space using the cameras.
pts = torch.zeros(len(cams), 1, 3)
print(cams.transform_points_screen(pts, image_size=(200, 200))[:3])

In [ ]:
import os
# List the files in the poster dataset directory
# Load and inspect the poster dataset
print(os.listdir("../data/poster"))
p_imgs, p_sils, p_cams = load_nerf_data("../data/poster", split="train")
print(p_imgs.shape)
print(p_cams.get_camera_center()[:3])

In [ ]:
import matplotlib.pyplot as plt
# Visualize the first image, silhouette, and poster image
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(imgs[0].numpy()); ax[1].imshow(sils[0].numpy()); ax[2].imshow(p_imgs[0].numpy())

In [ ]:
import torch, json
from pytorch3d.io import IO

p_imgs, p_sils, p_cams = load_nerf_data("../data/poster", split="train", resize_to=(320, 180))
pts = IO().load_pointcloud("../data/poster/sparse_pc.ply").points_packed()

# The loader applies applied_transform to the cameras, so apply it to the points too
with open("../data/poster/transforms.json") as f:
    t = json.load(f)
A = torch.tensor(t["applied_transform"], dtype=torch.float32)
pts = pts @ A[:3, :3].T + A[:3, 3]

depths, in_front = [], []
for i in range(0, len(p_cams), 20):
    z = p_cams[i].get_world_to_view_transform().transform_points(pts[None])[0, :, 2]
    in_front.append((z > 0).float().mean().item())
    depths.append(z[z > 0])
d = torch.cat(depths)
print("points:", len(pts), " fraction in front of camera:", sum(in_front) / len(in_front))
print("near (1%):", d.quantile(0.01).item(), " far (99%):", d.quantile(0.99).item())
print(d.quantile(torch.tensor([0.05, 0.5, 0.9, 0.95])))